# Manual Bakery review

This notebook reviews the bakery verification results from Notebook 08.

The purpose is to identify specific classifications that may require manual checking and to maintain a finalised manual-review record.

In [18]:
from pathlib import Path

import pandas as pd

VERIFICATION_FOLDER = Path("../data/business/interim/ai_verification_v2")

AI_RESULTS_PATH = (VERIFICATION_FOLDER / "bakery_ai_verification_results.csv")

MANUAL_REVIEW_FOLDER = (VERIFICATION_FOLDER / "manual_review")
MANUAL_REVIEW_FOLDER.mkdir(parents=True, exist_ok=True,)

MANUAL_REVIEW_PATH = (MANUAL_REVIEW_FOLDER / "bakery_manual_review.xlsx")

EXPECTED_TOTAL = 28847

## Load AI verification results

The AI verification results are loaded as the source dataset for manual-review diagnostics. Checks are completed as AI-verification occurs and thus a completion percentage helps keep track of how much has been been produced.

In [2]:
ai_results = pd.read_csv(AI_RESULTS_PATH)

ai_results = (ai_results
              .sort_values("BakeryRank")
              .reset_index(drop=True))

print(f"AI verification rows loaded: {len(ai_results)}")

print(f"Highest BakeryRank available: {ai_results['BakeryRank'].max()}")

print(f"Verification completion: {len(ai_results) / EXPECTED_TOTAL:.2%}")

AI verification rows loaded: 28847
Highest BakeryRank available: 28847
Verification completion: 100.00%


## Check loaded verification results

A small number of checks are used to make sure the verification results are suitable for manual review.

In [3]:
ESSENTIAL_COLUMNS = [
    "BusinessNameClean",
    "BusinessName",
    "FHRSIDRep",
    "BusinessType",
    "Address",
    "PostCode",
    "LocalAuthorityName",
    "BakeryRank",
    "BakeryScore",
    "StoreCount",
    "LocationMatch",
    "AIStatus",
    "PhysicalRetail",
    "BakeryFocus",
    "AIVerdict",
    "AIReason"]

missing_columns = [column
                   for column in ESSENTIAL_COLUMNS
                   if column not in ai_results.columns]

if missing_columns:
    raise ValueError(f"Missing expected columns: {missing_columns}")

if ai_results["BakeryRank"].isna().any():
    raise ValueError("Missing BakeryRank values found.")

if ai_results["BakeryRank"].duplicated().any():
    raise ValueError("Duplicate BakeryRank values found.")

if len(ai_results) != EXPECTED_TOTAL:
    raise ValueError(f"Expected {EXPECTED_TOTAL} verification results, found {len(ai_results)}.")

print("Verification results ready for review.")

Verification results ready for review.


## Manual review approach

After inspecting the ai verifiied CSV, there were a handful of specific cases where it was apparent further review would be needed. The manual review will target focussed groups where the business either shows a bakery-like identity but was classified otherwise or where the classifiction signals from the AI showed disagreements and thus may be worth reviewing further.

Final decisions from the review are entered manually.

In [4]:
verdict_summary = (ai_results["AIVerdict"]
                   .value_counts()
                   .rename("Count")
                   )

display(verdict_summary)

AIVerdict
NOT_BAKERY    19849
UNCLEAR        7048
BAKERY         1950
Name: Count, dtype: int64

## Identify potential false negatives and unresolved cases

With potential false negatives, the most obvious disagreement is where the business name contains bakery-related words but the AI verification classified the business as NOT BAKERY. Since we have the bakery focus signal, we can also check where the AI gave indication of bakery focus and physical retail locations however classified as NOT BAKERY.

Patterns found from parsing through AI Verification CSV

In [5]:
STRONG_BAKERY_NAME_PATTERN = (r"\b(?:bakery|bakeries|bakers|bakehouse|boulangerie|patisserie|pâtisserie|panaderia|panadería)\b")


WEAK_PHYSICAL_REASON_PATTERN = (r"\bno evidence|no clear evidence|no (?:clear )?indication\b")


MARKET_RETAIL_PATTERN = (r"\bmarkets?|stalls?|pop[- ]?ups?\b")

In [6]:
bakery_name_signal = (ai_results["BusinessNameClean"].fillna("")
                      .str.contains(STRONG_BAKERY_NAME_PATTERN, case=False, regex=True,))


bakery_like_name_disagreement = ai_results[(ai_results["AIVerdict"] != "BAKERY")
                                      & (ai_results["LocationMatch"] == "YES")
                                      & (ai_results["AIStatus"] == "ACTIVE")
                                      & (ai_results["PhysicalRetail"] != "NO")
                                      & (ai_results["BakeryFocus"] != "YES")
                                      & bakery_name_signal].copy()

bakery_like_name_disagreement["ReviewReason"] = ("Strong bakery-like name disagreement with unresolved/no bakery focus")

print(f"Bakery-like name disagreements: {len(bakery_like_name_disagreement)}")

Bakery-like name disagreements: 40


The prompt said if PhysicalRetail = No then it should be classified NOT BAKERY however a case was found where it was classified UNCLEAR even though the PhysicalRetail matched NO and so it may be worth checking for more such cases.

In [7]:
no_physical_retail_unclear_bakery = ai_results[(ai_results["AIVerdict"] == "UNCLEAR")
                                               & (ai_results["LocationMatch"] == "YES")
                                               & (ai_results["AIStatus"] == "ACTIVE")
                                               & (ai_results["PhysicalRetail"] == "NO")
                                               & (ai_results["BakeryFocus"] == "UNCLEAR")].copy()


no_physical_retail_unclear_bakery["ReviewReason"] = ("UNCLEAR despite PhysicalRetail=NO")

print(f"UNCLEAR despite PhysicalRetail=NO: {len(no_physical_retail_unclear_bakery)}")

UNCLEAR despite PhysicalRetail=NO: 31


In [8]:
physical_retail_unclear_focus = ai_results[(ai_results["AIVerdict"] == "UNCLEAR")
                                        & (ai_results["LocationMatch"] == "YES")
                                        & (ai_results["AIStatus"] == "ACTIVE")
                                        & (ai_results["PhysicalRetail"] == "YES") 
                                        & (ai_results["BakeryFocus"] == "UNCLEAR")].copy()


physical_retail_unclear_focus["ReviewReason"] = ("Physical retail confirmed; bakery focus unresolved")


print(f"PhysicalRetail YES / BakeryFocus UNCLEAR: {len(physical_retail_unclear_focus)}")

PhysicalRetail YES / BakeryFocus UNCLEAR: 196


In [9]:
unclear_physical_retail_bakery_focus = ai_results[(ai_results["AIVerdict"] == "UNCLEAR")
                                                  & (ai_results["LocationMatch"] == "YES")
                                                  & (ai_results["AIStatus"] == "ACTIVE")
                                                  & (ai_results["PhysicalRetail"] == "UNCLEAR") 
                                                  & (ai_results["BakeryFocus"] == "YES")].copy()

unclear_physical_retail_bakery_focus["ReviewReason"] = ("Bakery focus confirmed; physical retail unresolved")

print(f"PhysicalRetail UNCLEAR / BakeryFocus YES: {len(unclear_physical_retail_bakery_focus)}")

PhysicalRetail UNCLEAR / BakeryFocus YES: 42


Since "PhysicalRetail = NO" would lead to a negative AIVerdict, it is worth ensuring that the AIReason in those situations gave strong reasoning as to why it believes PhysicalRetail does not exist.

In [10]:
no_physical_retail_bakery_focus = ai_results[(ai_results["AIVerdict"] == "NOT_BAKERY")
                                   & (ai_results["LocationMatch"] == "YES")
                                   & (ai_results["AIStatus"] == "ACTIVE")
                                   & (ai_results["PhysicalRetail"] == "NO")
                                   & (ai_results["BakeryFocus"] == "YES")].copy()


weak_physical_reason = (no_physical_retail_bakery_focus["AIReason"].fillna("")
                        .str.contains(WEAK_PHYSICAL_REASON_PATTERN, case=False, regex=True))

market_retail_reason = (no_physical_retail_bakery_focus["AIReason"].fillna("")
                        .str.contains(MARKET_RETAIL_PATTERN, case=False, regex=True))

no_physical_retail_review = (no_physical_retail_bakery_focus[weak_physical_reason | market_retail_reason].copy())

no_physical_retail_review["ReviewReason"] = ("Bakery focus YES; PhysicalRetail NO with weak or market-based evidence")

print(f"Targeted PhysicalRetail=NO cases: {len(no_physical_retail_review)}")

Targeted PhysicalRetail=NO cases: 90


## Identify potential false positive cases

These checks will be made based on independent signals that may show disagreements with the positive classification.

In [11]:
bakery_rows = ai_results[ai_results["AIVerdict"] == "BAKERY"].copy()

print(f"AI-classified BAKERIES: {len(bakery_rows)}")

AI-classified BAKERIES: 1950


In [12]:
low_score_candidates = int(len(bakery_rows) * 0.10)

low_score_bakeries = (bakery_rows.nsmallest(low_score_candidates, "BakeryScore",).copy())

low_score_bakeries["ReviewReason"] = (f"AI BAKERY in lowest 10% of BakeryScore")

print(f"Lowest 10% BakeryScore candidates: {len(low_score_bakeries)}")

Lowest 10% BakeryScore candidates: 195


In [13]:
FHRS_UNUSUAL_BUSINESS_TYPES= ["Manufacturers/packers", 
                              "Mobile caterer", 
                              "Retailers - supermarkets/hypermarkets", 
                              "Hotel/bed & breakfast/guest house",
                              "Caring Premises",
                              "Distributors/Transporters",
                              "Pub/bar/nightclub",
                              "School/college/university"]

unusual_business_type_bakeries = bakery_rows[bakery_rows["BusinessType"]
                                             .isin(FHRS_UNUSUAL_BUSINESS_TYPES)].copy()

unusual_business_type_bakeries["ReviewReason"] = ("AI BAKERY with unusual FHRS BusinessType")

print(f"Unusual BusinessType Bakeries: {len(unusual_business_type_bakeries)}")

Unusual BusinessType Bakeries: 143


Another reasoning pattern with the AI verification that was suspicious and worth reviewing was with AIReasons containing the word "qualifying" as it showed times where the bakery definition has been taken too broadly.

In [14]:
qualifying_bakeries_reasoning = bakery_rows[bakery_rows["AIReason"].fillna("")
                                  .str.contains(r"\bqualifying\b", case=False, regex=True,)].copy()


qualifying_bakeries_reasoning["ReviewReason"] = ("AI BAKERY using explicit qualifying-product reasoning")


print(f"Explicit qualifying-product reasoning: {len(qualifying_bakeries_reasoning)}")

Explicit qualifying-product reasoning: 56


In [15]:
SUPERMARKET_GROCERY_TYPES = ["Retailers - other", "Retailers - supermarkets/hypermarkets"]


SUPERMARKET_GROCERY_PATTERN = (r"\b(?:supermarket|supermarkets|grocery|groceries|grocer|grocers)\b")

supermarket_grocery_bakeries = bakery_rows[(bakery_rows["BusinessType"].isin(SUPERMARKET_GROCERY_TYPES))
                                           & (bakery_rows["AIReason"].fillna("")
                                              .str.contains(SUPERMARKET_GROCERY_PATTERN, case=False, regex=True))].copy()

supermarket_grocery_bakeries["ReviewReason"] = ("AI BAKERY appears to be a general supermarket or grocery retailer")

print(f"Supermarket/grocery AI-classified BAKERIES: {len(supermarket_grocery_bakeries)}")

Supermarket/grocery AI-classified BAKERIES: 24


In [ ]:
manual_review_groups = [bakery_like_name_disagreement,
                        no_physical_retail_unclear_bakery,
                        physical_retail_unclear_focus,
                        unclear_physical_retail_bakery_focus,
                        no_physical_retail_review,
                        low_score_bakeries,
                        unusual_business_type_bakeries,
                        qualifying_bakeries_reasoning,
                        supermarket_grocery_bakeries]

complete_manual_review = pd.concat(manual_review_groups, ignore_index=True,)

review_reasons = (complete_manual_review.groupby("BakeryRank", as_index=False)["ReviewReason"].agg(" | ".join))

manual_review_reasoned = (ai_results[ai_results["BakeryRank"].isin(review_reasons["BakeryRank"])
    ].merge(review_reasons, on="BakeryRank", how="left",).copy())


if manual_review_reasoned["BakeryRank"].duplicated().any():
    raise ValueError("Duplicate BakeryRank values found in targeted review set.")

review_total = len(manual_review_reasoned)

print(f"Unique targeted manual-review rows: {review_total}")

display(manual_review_reasoned["AIVerdict"].value_counts().rename("Count"))

Unique targeted manual-review rows: 794


AIVerdict
BAKERY        397
UNCLEAR       277
NOT_BAKERY    120
Name: Count, dtype: int64

# Create manual-review workbook

In [20]:
manual_review_export = manual_review_reasoned.copy()

manual_review_export["ManualVerdict"] = ""
manual_review_export["ManualNote"] = ""

EXPORT_COLUMNS = ["BakeryRank",
                  "BusinessName",
                  "BusinessNameClean",
                  "BusinessType",
                  "Address",
                  "PostCode",
                  "LocalAuthorityName",
                  "FHRSIDRep",
                  "StoreCount",
                  "ManualVerdict",
                  "ManualNote",
                  "BakeryScore",
                  "AIVerdict",
                  "LocationMatch",
                  "AIStatus",
                  "PhysicalRetail",
                  "BakeryFocus",
                  "AIReason",
                  "ReviewReason"]

manual_review_export = manual_review_export[EXPORT_COLUMNS]

print(f"Workbook rows prepared: {len(manual_review_export)}")

Workbook rows prepared: 794


In [37]:
if MANUAL_REVIEW_PATH.exists():
    print("Manual review workbook already exists.")
    print("Existing workbook was NOT overwritten.")
    print(MANUAL_REVIEW_PATH)

else:
    manual_review_export.to_excel(MANUAL_REVIEW_PATH, 
                                  sheet_name="ManualReview", 
                                  index=False, 
                                  freeze_panes=(1, 0))
    
    print(f"Manual review workbook created: {MANUAL_REVIEW_PATH}")

Manual review workbook already exists.
Existing workbook was NOT overwritten.
..\data\business\interim\ai_verification_v2\manual_review\bakery_manual_review.xlsx


In [45]:
if not MANUAL_REVIEW_PATH.exists():
    print("Manual review workbook has not been created yet.")

else:
    review_progress = pd.read_excel(MANUAL_REVIEW_PATH, sheet_name="ManualReview",)

    print(f"Manual review workbook loaded: {len(review_progress)} rows")

Manual review workbook loaded: 794 rows


In [46]:
if MANUAL_REVIEW_PATH.exists():
    total_reviews = len(review_progress)

    valid_manual_verdicts = ["BAKERY", "NOT_BAKERY", "UNCLEAR", "MIXED"]

    completed_manual_reviews = review_progress["ManualVerdict"].isin(valid_manual_verdicts)

    completed_reviews = int(completed_manual_reviews.sum())

    remaining_reviews = (total_reviews - completed_reviews)

    completion_percentage = (completed_reviews / total_reviews)

    print(f"Total review rows: {total_reviews}")
    print(f"Completed: {completed_reviews}")
    print(f"Remaining: {remaining_reviews}")
    print(f"Review completion: {completion_percentage:.1%}")

Total review rows: 794
Completed: 150
Remaining: 644
Review completion: 18.9%


In [47]:
if MANUAL_REVIEW_PATH.exists():
    manual_verdict_counts = (
        review_progress.loc[completed_manual_reviews, "ManualVerdict"]
        .value_counts()
        .rename("Count"))


    display(manual_verdict_counts)

ManualVerdict
NOT_BAKERY    75
BAKERY        74
MIXED          1
Name: Count, dtype: int64